# Synthetic image generation with Gemini

This notebook generates **synthetic images** for the **Urban Disaster Monitor** dataset using the **Gemini 2.5 Flash Image** model. The images simulate disaster scenarios (floods) with animals (cows, horses, cats, dogs) in a documentary/photojournalistic style, to enrich the dataset and reduce class imbalance.

---
**Project:** [Urban Disaster Monitor](https://github.com/MariaCarolinass/urban-disaster-monitor) · **Authors:** Carolina Soares, João Galdino

## Generate images in the web interface

1. Open [Gemini 2.5 Flash Image](https://ai.google.com/research/gemini).
2. Enter the prompts in the image generation section.
3. Adjust resolution and style as needed.
4. Generate and download the images for inclusion in the dataset.

## Generate images with the API

The cells below use the **Google Generative AI API** to generate multiple images in sequence. You need an API key (Google AI Studio or Google Cloud).

### Setup and dependencies

Install `app/requirements.txt` and `notebooks/requirements.txt` in the kernel environment.

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "dataset" / "data.yaml").is_file() and (p / "models").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project repository.")
import yaml
with (PROJECT_ROOT / "dataset" / "data.yaml").open() as file:
    dataset_config = yaml.safe_load(file)
CLASS_NAMES = dataset_config["names"]
if isinstance(CLASS_NAMES, dict):
    CLASS_NAMES = [CLASS_NAMES[i] for i in sorted(CLASS_NAMES)]
print(f"Dataset classes: {CLASS_NAMES}")

Get an API key from [Google AI Studio](https://aistudio.google.com/apikey) or [Google Cloud Console](https://console.cloud.google.com/). Set the `GOOGLE_API_KEY` environment variable or use the code below with `getpass` to enter the key securely.

### Run batch generation

Prompts are built from the classes in `dataset/data.yaml`. Generated images are saved in `notebooks/generated/images/`.

In [ ]:
import os
import getpass
api_key = os.environ.get("GOOGLE_API_KEY") or getpass.getpass("GOOGLE_API_KEY: ")
if not api_key:
    raise ValueError("GOOGLE_API_KEY is required to generate images.")

In [ ]:
from google import genai
from PIL import Image
from io import BytesIO
import time

client = genai.Client(api_key=api_key)
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "generated" / "images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The subjects come from the project's dataset; scene/style is an experiment parameter.
SUBJECT_NAMES = {"civilian": "a civilian", "rescuer": "a rescue worker"}
prompts = [
    f"Realistic documentary photograph of {SUBJECT_NAMES.get(name, 'a ' + name)} "
    "in an urban flood rescue scene, natural lighting, visible full body, realistic proportions"
    for name in CLASS_NAMES
]
MODEL_NAME = "gemini-2.5-flash-image"
MAX_RETRIES = 2
DELAY_BETWEEN_REQUESTS = 1.0

def generate_and_save_image(prompt, index, total):
    print(f"[{index}/{total}] {prompt[:60]}...")
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
            for candidate in response.candidates or []:
                for part in candidate.content.parts:
                    if part.inline_data:
                        image = Image.open(BytesIO(part.inline_data.data))
                        filename = OUTPUT_DIR / f"{CLASS_NAMES[index - 1]}_{index:02d}.png"
                        image.save(filename)
                        print(f"Saved: {filename}")
                        return True
        except Exception as exc:
            print(f"Attempt {attempt + 1}/{MAX_RETRIES} failed: {exc}")
        time.sleep(DELAY_BETWEEN_REQUESTS)
    return False

for index, prompt in enumerate(prompts, start=1):
    generate_and_save_image(prompt, index, len(prompts))
    time.sleep(DELAY_BETWEEN_REQUESTS)
print(f"Images in: {OUTPUT_DIR}")

### Review generated files

Run the cell below to list files in the output folder.

In [ ]:
files = sorted(OUTPUT_DIR.glob("*.png"))
print(f"Total: {len(files)} images")
for path in files[:15]:
    print(f"  {path.name}")
if len(files) > 15:
    print(f"  ... and {len(files) - 15} more")